# GeoCLIP baseline on Kaggle

This notebook loads the repository's pretrained GeoCLIP model and fixed 100K GPS gallery, then runs the bundled `Kauai.png` smoke sample. The Kalalau Lookout coordinate is an approximate smoke reference, so this is not a standardized benchmark score.

In [1]:
import os, sys, time
from pathlib import Path
import torch

base = Path('/kaggle/working/geoclip_baseline')
source_dir = base / 'source'
hf_root = base / 'hf'
model_file = hf_root / 'hub/models--openai--clip-vit-large-patch14/snapshots/local/model.safetensors'
assert source_dir.is_dir(), f'Missing GeoCLIP source: {source_dir}'
assert model_file.is_file(), f'Missing CLIP backbone: {model_file}'

os.environ.update(
    HF_HOME=str(hf_root),
    HF_HUB_OFFLINE='1',
    TRANSFORMERS_OFFLINE='1',
    TOKENIZERS_PARALLELISM='false',
)
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from geoclip import GeoCLIP

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
model = GeoCLIP().to(device).eval()
if device == 'cuda':
    torch.cuda.synchronize()
model_load_seconds = time.perf_counter() - start
print('BASELINE_SETUP_OK')
print('device:', device)
print('gpu:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
print('gallery_size:', int(model.gps_gallery.shape[0]))
print('model_load_seconds:', round(model_load_seconds, 4))

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


BASELINE_SETUP_OK
device: cuda
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
gallery_size: 100000
model_load_seconds: 0.7502


In [2]:
import json, time
from geopy.distance import geodesic

image_path = source_dir / 'geoclip/images/Kauai.png'
assert image_path.is_file(), image_path
reference = (22.1538889, -159.6483333)

start = time.perf_counter()
top_gps, top_prob = model.predict(str(image_path), top_k=5)
if device == 'cuda':
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - start

predictions = []
for rank, (gps, probability) in enumerate(zip(top_gps.tolist(), top_prob.tolist()), start=1):
    predictions.append({
        'rank': rank,
        'lat': gps[0],
        'lon': gps[1],
        'probability': probability,
        'distance_to_reference_km': geodesic(reference, gps).km,
    })

top1_km = predictions[0]['distance_to_reference_km']
thresholds = (1, 25, 200, 750, 2500)
result = {
    'sample': 'Kauai.png',
    'reference_kind': 'approximate Kalalau Lookout smoke reference',
    'reference_gps': reference,
    'device': device,
    'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else None,
    'gallery_size': int(model.gps_gallery.shape[0]),
    'model_load_seconds': model_load_seconds,
    'inference_seconds': inference_seconds,
    'top1_distance_km': top1_km,
    'accuracy_single_sample': {
        f'acc_{distance}_km': float(top1_km <= distance)
        for distance in thresholds
    },
    'top5': predictions,
    'peak_gpu_gib': torch.cuda.max_memory_allocated() / 1024**3 if device == 'cuda' else 0.0,
}
output_path = base / 'baseline_local_notebook_result.json'
output_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
print('BASELINE_LOCAL_NOTEBOOK_OK')
print(json.dumps(result, indent=2))
print('saved:', output_path)

BASELINE_LOCAL_NOTEBOOK_OK
{
  "sample": "Kauai.png",
  "reference_kind": "approximate Kalalau Lookout smoke reference",
  "reference_gps": [
    22.1538889,
    -159.6483333
  ],
  "device": "cuda",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "gallery_size": 100000,
  "model_load_seconds": 0.7502205620003224,
  "inference_seconds": 0.29954881699995894,
  "top1_distance_km": 5.591105931817587,
  "accuracy_single_sample": {
    "acc_1_km": 0.0,
    "acc_25_km": 1.0,
    "acc_200_km": 1.0,
    "acc_750_km": 1.0,
    "acc_2500_km": 1.0
  },
  "top5": [
    {
      "rank": 1,
      "lat": 22.197973251342773,
      "lon": -159.6219024658203,
      "probability": 0.0725473165512085,
      "distance_to_reference_km": 5.591105931817587
    },
    {
      "rank": 2,
      "lat": 22.178503036499023,
      "lon": -159.65005493164062,
      "probability": 0.07002350687980652,
      "distance_to_reference_km": 2.7313636734633255
    },
    {
      "rank": 3,
      "lat": 22.175880432